In [3]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
print(dataset)
print(dataset["train"][0])

/Users/mac/Code/Python Projects/ai-practice/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [4]:
dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

for i, example in enumerate(dataset):
    print(example["title"])
    if i >= 4:
        break

Anarchism
Albedo
A
Alabama
Achilles


In [6]:
print(type(dataset))

<class 'datasets.iterable_dataset.IterableDataset'>


In [7]:
dataset = load_dataset("stanfordnlp/imdb", split="train")

dataset.to_csv("imdb_train.csv")
dataset.to_json("imdb_train.json")
dataset.to_parquet("imdb_train.parquet")

Creating parquet from Arrow format: 100%|██████| 1/1 [00:00<00:00,  3.26ba/s]


33432823

In [ ]:
import os

for f in ["imdb_train.csv", "imdb_train.json", "imdb_train.parquet"]:
    print(f, round(os.path.getsize(f) / 1e6, 1), "MB")

imdb_train.csv 33.3 MB
imdb_train.json 33.9 MB
imdb_train.parquet 20.5 MB


In [9]:
for f in ["imdb_train.csv", "imdb_train.json", "imdb_train.parquet"]:
    os.remove(f)

In [10]:
dataset = load_dataset("stanfordnlp/imdb", split="train")

split = dataset.train_test_split(test_size=0.2, seed=42)
train_val = split["train"].train_test_split(test_size=0.125, seed=42)

train_ds = train_val["train"]
val_ds = train_val["test"]
test_ds = split["test"]

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Train: 17500, Val: 2500, Test: 5000


In [11]:
print(train_ds[0]["text"][:80])

A really bad sequel. Part 1 had a lot of funny moments - part 2 is just bad (in 


In [12]:
from huggingface_hub import hf_hub_download, snapshot_download

model_path = hf_hub_download(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    filename="config.json"
)
print(f"Cached at: {model_path}")

Cached at: /Users/mac/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json


In [13]:
model_dir = snapshot_download(
    "sentence-transformers/all-MiniLM-L6-v2",
    allow_patterns=["*.json", "*.txt", "model.safetensors"],
)
print(f"Full model at: {model_dir}")

Fetching 11 files: 100%|█████████████████████| 11/11 [00:10<00:00,  1.06it/s]

Full model at: /Users/mac/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/1110a243fdf4706b3f48f1d95db1a4f5529b4d41


In [14]:
import os

local_path = os.path.expanduser("~/.cache/huggingface/datasets/")
print(local_path)
print(os.listdir(local_path))

/Users/mac/.cache/huggingface/datasets/
['_Users_mac_.cache_huggingface_datasets_wikimedia___wikipedia_20231101.en_0.0.0_b04c8d1ceb2f5cd4588862100d08de323dccfbaa.lock', '_Users_mac_.cache_huggingface_datasets_stanfordnlp___imdb_plain_text_0.0.0_e6281661ce1c48d982bc483cf8a173c1bbeb5d31.lock', 'stanfordnlp___imdb']


In [16]:
ds = load_dataset("nyu-mll/glue", "mrpc")
print(ds["train"][:5])

Generating train split: 100%|██| 3668/3668 [00:00<00:00, 59884.58 examples/s]
Generating validation split: 100%|█| 408/408 [00:00<00:00, 31398.98 examples/
Generating test split: 100%|███| 1725/1725 [00:00<00:00, 98943.91 examples/s]

{'sentence1': ['Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .', "Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .", 'They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .', 'Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .', 'The stock rose $ 2.11 , or about 11 percent , to close Friday at $ 21.51 on the New York Stock Exchange .'], 'sentence2': ['Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .', "Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .", "On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .", 'Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .', 'P

In [17]:
import time

ds = load_dataset("allenai/c4", "en", split="train", streaming=True)
start, count = time.time(), 0
for _ in ds:
    count += 1
    if time.time() > start + 10:
        break
print(count)

32054


In [18]:
import os

imdb = load_dataset("stanfordnlp/imdb", split="train")
imdb.to_parquet("imdb_train.parquet")

reloaded = load_dataset("parquet", data_files="imdb_train.parquet", split="train")
print(len(imdb), len(reloaded))
print(imdb[0]["text"][:60] == reloaded[0]["text"][:60])

os.remove("imdb_train.parquet")

Creating parquet from Arrow format: 100%|██████| 1/1 [00:00<00:00,  1.59ba/s]
Generating train split: 25000 examples [00:00, 41518.74 examples/s]

25000 25000
True


In [19]:
dataset = load_dataset("stanfordnlp/imdb", split="train")

a = dataset.train_test_split(test_size=0.3, seed=42)
b = a["test"].train_test_split(test_size=0.5, seed=42)

train_ds = a["train"]
val_ds = b["train"]
test_ds = b["test"]

print(len(train_ds), len(val_ds), len(test_ds))
print(len(train_ds) + len(val_ds) + len(test_ds) == len(dataset))

17500 3750 3750
True


In [20]:
print(train_ds[0]["text"][:60])
print(val_ds[0]["text"][:60])
print(test_ds[0]["text"][:60])

One question that must be asked immediately is: Would this f
This is one of those films that explore the culture clash of
If this film had a budget of 20 million I'd just like to kno
